# FabricOps Guardrail maintainer validation

One disposable notebook, using the **installed FabricOps Starter Kit**. No mocks, demo data, UI authoring, or repository checkout is required. Internal maintainer artifact; remove this file without affecting the package.

**Execution requires Microsoft Fabric Spark and existing Development Bronze, Silver and Metadata Lakehouses.** Fill in the three store coordinates below (or reuse an already initialized standard `FABRIC_CONTEXT`). Metadata must already have been provisioned using the standard environment setup. This notebook checks prerequisites; it does not provision shared metadata or activate contracts.

Run sections 00–08 in order. Section 09 is intentionally inert during Run all: after reviewing results, paste the printed run namespace into its cleanup switch and explicitly run that cell. Keep this session and its ownership manifest until cleanup completes. A new session/run uses new table identities.

Runtime validation: **NOT RUN in Microsoft Fabric**. Local validation can establish notebook structure, Python syntax and package compatibility only. Expected failures are assertions about runtime behavior, never fabricated PASS results. Frozen payloads, raw orchestrator returns, exceptions, stage logs and scoped `METADATA_GUARDRAIL_RESULTS` are retained in `ARTIFACTS` for investigation.


## 00. Environment

Use standard `FrameworkConfig`, `FabricStore`, `PathConfig` and `setup_notebook`. Standalone configuration below uses only `dev`. Set the schema flags to match the existing Lakehouses. When a standard context is already present, its configuration is reused and strictly checked for Development. No `%run` dependency on another notebook.


In [ ]:
import builtins
import copy
import io
import json
import re
import traceback
import uuid
from contextlib import redirect_stdout, redirect_stderr
from datetime import datetime, timedelta

from IPython import display as ip
from pyspark.sql import SparkSession, functions as F
from delta.tables import DeltaTable
from fabricops_kit import (
    __version__, FabricStore, FrameworkConfig, PathConfig, setup_notebook,
    orchestrate_read, orchestrate_write, profile_table, resolve_table_id,
    read_lakehouse_table, write_lakehouse_table,
)
from fabricops_kit.config.shared import get_default_fabric_context, get_store
from fabricops_kit.config.audit import build_runtime_audit_fields
from fabricops_kit.config.metadata_schemas import (
    metadata_table_physical_schema, metadata_table_schema_registry,
)
from fabricops_kit.io.shared import resolve_configured_lakehouse_table
# Existing UI-independent authoritative services; deliberately internal maintainer usage.
from fabricops_kit.data_contract.shared import (
    create_contract_draft, save_contract_draft, freeze_contract,
    canonical_guardrail_rule_record,
)

# Standalone bootstrap: supply YOUR DEVELOPMENT coordinates only. Never commit real IDs.
DEVELOPMENT_STORES = {
    "Bronze": {"workspace_id": "", "item_id": "", "schema_enabled": True, "schema": "dbo"},
    "Silver": {"workspace_id": "", "item_id": "", "schema_enabled": True, "schema": "dbo"},
    "Metadata": {"workspace_id": "", "item_id": "", "schema_enabled": True, "schema": "governance"},
}
try:
    ORIGINAL_CONTEXT = get_default_fabric_context()
except RuntimeError:
    ORIGINAL_CONTEXT = None
if ORIGINAL_CONTEXT is None:
    missing = [key for key, value in DEVELOPMENT_STORES.items()
               if not value["workspace_id"] or not value["item_id"]]
    if missing:
        raise RuntimeError("Configure Development Lakehouse coordinates in DEVELOPMENT_STORES: " + ", ".join(missing))
    CONFIG = FrameworkConfig(path_config=PathConfig(paths={
        "dev": {key: FabricStore(env="dev", kind="lakehouse", **value)
                for key, value in DEVELOPMENT_STORES.items()}
    }))
    ENV = "dev"
else:
    CONFIG, ENV = ORIGINAL_CONTEXT["config"], ORIGINAL_CONTEXT["env"]
if ENV != "dev":
    raise RuntimeError(f"Development-only notebook: expected env='dev', found {ENV!r}.")
spark = SparkSession.getActiveSession()
if spark is None:
    raise RuntimeError("An active Microsoft Fabric Spark session is required.")
spark.conf.set("spark.sql.session.timeZone", CONFIG.audit_timezone)
STORES = {key: get_store(CONFIG, ENV, key) for key in ("Bronze", "Silver", "Metadata")}
for key, store in STORES.items():
    if store.env != "dev" or store.kind != "lakehouse":
        raise RuntimeError(f"{key} must be a configured Development Lakehouse.")
    # Refuse a Development data store that aliases a configured Production data store.
    if key != "Metadata":
        for env_key, env_stores in CONFIG.path_config.paths.items():
            if str(env_key).lower() in {"prod", "production"}:
                for prod in env_stores.values():
                    if (store.workspace_id, store.item_id) == (prod.workspace_id, prod.item_id):
                        raise RuntimeError(f"{key} aliases a configured Production item; refusing mutations.")
for key, store in STORES.items():
    hroot = spark._jvm.org.apache.hadoop.fs.Path(store.root + "/Tables")
    try:
        assert hroot.getFileSystem(spark._jsc.hadoopConfiguration()).exists(hroot)
    except Exception as exc:
        raise RuntimeError(f"Required Development Lakehouse unavailable: {key}") from exc
RUN_CONTEXT = setup_notebook(CONFIG, env=ENV, required_targets=list(STORES))
# The requested disposable 04_ name is outside the standard template naming policy.
# Retain that diagnostic and waive ONLY naming; all environment failures still stop execution.
failed_checks = [c for c in RUN_CONTEXT.validation_results if c.status == "fail" and c.name != "notebook_naming"]
if failed_checks:
    raise RuntimeError(f"Fabric prerequisites unavailable: {failed_checks}")
print("Standard setup checks:", RUN_CONTEXT.validation_results)
BASE_CONTEXT = dict(ORIGINAL_CONTEXT or {
    "config": CONFIG, "env": ENV, "runtime_metadata": RUN_CONTEXT.runtime_metadata,
})
FABRIC_CONTEXT = dict(BASE_CONTEXT)
builtins.FABRIC_CONTEXT = FABRIC_CONTEXT
# Fail before seeding when audit identity or routed metadata is unavailable.
build_runtime_audit_fields(config=CONFIG, env=ENV, runtime_context=FABRIC_CONTEXT)
META_NAMES = (
    "METADATA_DATA_CATALOGUE", "METADATA_DATA_PROFILED", "METADATA_DATA_PROFILED_FREQUENCY",
    "METADATA_DATA_LINEAGE", "METADATA_DATA_CONTRACT", "METADATA_ENRICHMENT",
    "METADATA_GUARDRAIL", "METADATA_GUARDRAIL_RESULTS", "METADATA_SOURCE_OBSERVATION",
)
def metadata(name):
    return read_lakehouse_table(name, store="Metadata",
        schema=metadata_table_physical_schema(CONFIG, name), spark_session=spark,
        context=FABRIC_CONTEXT)
for name in META_NAMES:
    try:
        actual = set(metadata(name).columns)
        required = set(metadata_table_schema_registry()[name].fieldNames())
        assert required <= actual, f"missing columns: {sorted(required - actual)}"
    except Exception as exc:
        raise RuntimeError(f"Required configured Metadata table unavailable/incompatible: {name}") from exc

# Do not reset this manifest by rerunning section 00 in the same session.
if globals().get("OWNED") and not globals().get("CLEANED", False):
    raise RuntimeError("This session already owns a test run. Review/clean it before starting another.")
RUN_NAMESPACE = "fo_gr_ut_" + uuid.uuid4().hex[:16]
OWNED, CONTRACTS, RESULTS, ARTIFACTS, PLAN = {}, {}, {}, {}, {}
CLEANED = False
# Fixed records/timestamps; broad clean freshness window keeps fixtures reproducible across years.
FIXED_TIME = datetime(2025, 1, 15, 12, 0, 0)
CLEAN_MAX_AGE_DAYS = 36500
print(f"FabricOps {__version__}; Fabric runtime validation pending; namespace={RUN_NAMESPACE}")
print("Cleanup confirmation value:", RUN_NAMESPACE)


## 01. Seed test data

Stable record IDs and timestamps; every mutation starts from `clean_frame()`. Source and target tables are reserved with `errorifexists` before any overwrite. All physical mutations check the in-memory ownership manifest. Registration uses the installed `profile_table()` service; no Catalogue rows are injected. DataFrames contain synthetic values only.


In [ ]:
FIXTURE_SCHEMA = "row_id int, partition_day date, modified_at timestamp, name string, category string, amount double, ceiling double, code string, required_note string, email string, age int, secret string"
FIXTURE_ROWS = [
    (1, FIXED_TIME.date(), FIXED_TIME, "Alice", "A", 20.0, 100.0, "AA01", "ready", "alpha@example.test", 25, "private-one"),
    (2, FIXED_TIME.date(), FIXED_TIME, "Bob", "B", 40.0, 100.0, "BB02", "ready", "beta@example.test", 45, "private-two"),
    (3, FIXED_TIME.date(), FIXED_TIME, "Cara", "A", 60.0, 100.0, "CC03", "ready", "alpha@example.test", 65, "private-three"),
]
def clean_frame():
    return spark.createDataFrame(FIXTURE_ROWS, FIXTURE_SCHEMA)

def reserve(role, store):
    name = RUN_NAMESPACE + "_" + role
    schema = STORES[store].schema if STORES[store].schema_enabled else None
    table_id = resolve_table_id(store=store, schema=schema, table_name=name)
    _, _, _, path = resolve_configured_lakehouse_table(store, name, schema, context=FABRIC_CONTEXT)
    hpath = spark._jvm.org.apache.hadoop.fs.Path(path)
    fs = hpath.getFileSystem(spark._jsc.hadoopConfiguration())
    if fs.exists(hpath):
        raise RuntimeError(f"Collision: test destination already exists for {role}; no overwrite allowed.")
    for metadata_name in ("METADATA_DATA_CATALOGUE", "METADATA_DATA_CONTRACT"):
        if metadata(metadata_name).where((F.col("environment_name") == ENV) &
                                         (F.col("table_id") == table_id)).limit(1).count():
            raise RuntimeError(f"Metadata identity collision for {role}; refusing to claim existing rows.")
    # Track the exact identity before writing, including partial seed failures.
    OWNED[role] = dict(name=name, store=store, schema=schema, table_id=table_id,
                       path=path, registered=False)
    write_lakehouse_table(clean_frame(), name, store=store, schema=schema,
                         mode="errorifexists", context=FABRIC_CONTEXT, verbose=False)
    profile_table(dataframe=clean_frame(), store=store, schema=schema, table_name=name,
                  frequency_enabled=False, spark_session=spark)
    OWNED[role]["registered"] = True
    return OWNED[role]

def put(role, frame):
    spec = OWNED[role]
    assert spec["name"].startswith(RUN_NAMESPACE + "_") and ENV == "dev"
    write_lakehouse_table(frame, spec["name"], store=spec["store"], schema=spec["schema"],
                         mode="overwrite", options={"overwriteSchema": "true"},
                         context=FABRIC_CONTEXT, verbose=False)

def physical(role):
    spec = OWNED[role]
    return read_lakehouse_table(spec["name"], store=spec["store"], schema=spec["schema"],
                               spark_session=spark, context=FABRIC_CONTEXT)

def change(frame, column, value, row_id=1):
    # Explicit cast preserves the schema while changing just the intended data value.
    dtype = dict(frame.dtypes)[column]
    return frame.withColumn(column, F.when(F.col("row_id") == row_id, F.lit(value).cast(dtype))
                            .otherwise(F.col(column)))

SOURCE = reserve("source", "Bronze")
TARGET = reserve("target", "Silver")
# A separate source has no Read evidence during coverage-negative cases.
UNREAD = reserve("unread", "Bronze")
COLUMN_IDS = {}
for role, spec in OWNED.items():
    rows = metadata("METADATA_DATA_CATALOGUE").where(
        (F.col("environment_name") == ENV) & (F.col("table_id") == spec["table_id"])
    ).collect()
    COLUMN_IDS[role] = {r["column_name"]: r["column_id"] for r in rows if r["column_id"]}
    assert set(clean_frame().columns) <= set(COLUMN_IDS[role]), f"Registration incomplete: {role}"
print("Reserved and registered:", {role: spec["name"] for role, spec in OWNED.items()})


## 02. Seed complete Data Contracts

All versions go through `create_contract_draft()` → `save_contract_draft()` → `freeze_contract()`. Rule definitions are normalized with the authoritative authoring service before being included in the draft payload. No frozen record is edited and no agreement/activation record is created. Test selection uses the same `data_contract_overrides` runtime context as the selector widget.

Separate versions isolate rule conflicts. Source contracts always include a legitimate Freshness rule because Read requires it; target contracts always include Schema because Write requires it. Sensitive Data failure fixtures use Schema monitoring so the intended treatment failure reaches the Sensitive Data stage. Coverage is an unconditional pre-publication gate: it has no configurable Warn action. Read has no Sensitive Data, Source Drift, or Coverage stage; Freshness has no Write stage.


In [ ]:
def rule(kind, rule_type, parameters=None, column=None):
    return dict(kind=kind, rule_type=rule_type, parameters=parameters or {}, column=column)

SCHEMA = rule("schema", "strict")
FRESH = rule("freshness", "max_age", {
    "freshness_column": "modified_at", "maximum_age": CLEAN_MAX_AGE_DAYS, "maximum_age_unit": "days"})
DRIFT = rule("source_drift", "historical_mutation", {
    "partition_column": "partition_day", "change_column": "modified_at"})
DQ_DEFS = {
    "Completeness": rule("data_quality", "completeness", {"columns": ["name"], "maximum_missing_percent": 0, "treat_blank_as_missing": True}),
    "Uniqueness": rule("data_quality", "uniqueness", {"columns": ["row_id"], "minimum_unique_percent": 100}),
    "Whitelist": rule("data_quality", "value_set", {"columns": ["category"], "mode": "allow", "values": ["A", "B"]}),
    "Blacklist": rule("data_quality", "value_set", {"columns": ["category"], "mode": "block", "values": ["DENY"]}),
    "Range": rule("data_quality", "range", {"columns": ["amount"], "minimum": 0, "maximum": 100, "minimum_inclusive": True, "maximum_inclusive": True}),
    "Pattern": rule("data_quality", "pattern", {"columns": ["code"], "pattern": "^[A-Z]{2}[0-9]{2}$"}),
    "Column Relationship": rule("data_quality", "column_relationship", {"columns": ["amount", "ceiling"], "operator": "<="}),
    "Conditional Completeness": rule("data_quality", "conditional_completeness", {"columns": ["category", "required_note"], "condition_operator": "=", "condition_value": "A", "treat_blank_as_missing": True}),
    "Conditional Values": rule("data_quality", "conditional_values", {"columns": ["category", "code"], "condition_operator": "=", "condition_value": "A", "mode": "allow", "values": ["AA01", "CC03"]}),
    "Custom Expression": rule("data_quality", "custom_expression", {"columns": ["amount"], "expression_language": "pyspark", "expression": "F.col('amount') >= F.lit(0)"}),
}
SENSITIVE = {
    "Mask": rule("sensitive_data", "mask", {"scope": "column", "treatment": "mask", "preserve_start": 1, "preserve_end": 1, "mask_character": "*"}, "email"),
    "Tokenize": rule("sensitive_data", "tokenize", {"scope": "column", "treatment": "tokenize"}, "email"),
    "Bucket": rule("sensitive_data", "bucket", {"scope": "column", "treatment": "bucket", "bins": [0, 30, 60], "labels": ["young", "middle", "older"]}, "age"),
    "Remove": rule("sensitive_data", "remove", {"scope": "column", "treatment": "remove"}, "secret"),
}

def seed_contract(key, role, definitions, action="Block", processing=None):
    if key in CONTRACTS:
        return CONTRACTS[key]
    table_id = OWNED[role]["table_id"]
    kwargs = dict(config=CONFIG, env=ENV, spark_session=spark, context=FABRIC_CONTEXT,
                  schema=metadata_table_physical_schema(CONFIG, "METADATA_DATA_CONTRACT"))
    draft = create_contract_draft(table_id=table_id, **kwargs)
    payload = json.loads(draft["contract_payload_json"])
    if processing:
        payload["table"]["processing"] = processing
    rules = []
    for index, definition in enumerate(definitions):
        rid = str(uuid.uuid5(uuid.NAMESPACE_URL, f"{RUN_NAMESPACE}/{key}/{index}"))
        raw = dict(contract_id=draft["contract_id"], contract_version=draft["contract_version"],
                   table_id=table_id, guardrail_rule_id=rid, guardrail_version=1,
                   rule_id=rid, guardrail_type=definition["kind"], rule_type=definition["rule_type"],
                   column_id=COLUMN_IDS[role].get(definition.get("column")),
                   rule_parameters_json=json.dumps(definition["parameters"]), action=action, is_active=True)
        # Preserve the authoring service shape: do not invent a severity field to
        # compensate for any runtime defect in translating Warn/Block actions.
        normalized = canonical_guardrail_rule_record(raw, config=CONFIG, env=ENV)
        doc = {name: normalized.get(name) for name in (
            "guardrail_rule_id", "guardrail_version", "contract_id", "contract_version",
            "column_id", "guardrail_type", "rule_id", "rule_type", "action", "is_active")}
        doc["rule_parameters"] = json.loads(normalized["rule_parameters_json"])
        rules.append(doc)
    payload["guardrails"] = rules
    saved = save_contract_draft(draft=draft, payload=payload, **kwargs)
    frozen = freeze_contract(draft=saved, **kwargs)
    record = frozen["contract"]
    assert record["status"] == "frozen" and not record["is_active"]
    assert frozen["payload"]["guardrails"] == rules, "Freeze changed authored rules"
    spec = dict(role=role, record=record, payload=frozen["payload"], rules=rules,
                contract_id=record["contract_id"], contract_version=int(record["contract_version"]))
    CONTRACTS[key] = spec
    return spec

BASE_SOURCE = seed_contract("baseline_source", "source", [SCHEMA, FRESH, DRIFT, DQ_DEFS["Completeness"]],
                            processing={"load_strategy": "append"})
BASE_TARGET = seed_contract("baseline_target", "target", [SCHEMA, DQ_DEFS["Completeness"]])
UNREAD_CONTRACT = seed_contract("unread", "unread", [SCHEMA, FRESH, DRIFT])

# Scenario vocabulary is local data, not a new package testing API.
CASES = []
def case(label, definition, mutate, boundaries=("Read", "Write"), schema_mode="strict", expected="violation"):
    CASES.append(dict(label=label, definition=definition, mutate=mutate,
                      boundaries=boundaries, schema_mode=schema_mode, expected=expected))

MUTATIONS = {
    "Completeness": lambda d: change(d, "name", None),
    "Uniqueness": lambda d: d.withColumn("row_id", F.when(F.col("row_id") == 2, F.lit(1)).otherwise(F.col("row_id"))),
    "Whitelist": lambda d: change(d, "category", "X"),
    "Blacklist": lambda d: change(d, "category", "DENY"),
    "Range": lambda d: change(d, "amount", -1.0),
    "Pattern": lambda d: change(d, "code", "invalid"),
    "Column Relationship": lambda d: change(d, "amount", 101.0),
    "Conditional Completeness": lambda d: change(d, "required_note", None),
    "Conditional Values": lambda d: change(d, "code", "ZZ99"),
    "Custom Expression": lambda d: change(d, "amount", -1.0),
}
for label, definition in DQ_DEFS.items():
    case(label, definition, MUTATIONS[label])
case("Completeness blank", DQ_DEFS["Completeness"], lambda d: change(d, "name", "  "))
case("Completeness null-only", rule("data_quality", "completeness", {"columns": ["name"], "maximum_missing_percent": 0, "treat_blank_as_missing": False}), lambda d: change(d, "name", None))
case("Uniqueness composite", rule("data_quality", "uniqueness", {"columns": ["category", "code"], "minimum_unique_percent": 100}), lambda d: change(d, "code", "AA01", row_id=3))
for field, value in (("minimum", 20.0), ("maximum", 60.0)):
    params = {"columns": ["amount"], field: value, field + "_inclusive": False}
    # Clean rows are moved away from the exclusive bound; mutation hits it exactly.
    # Dedicated fixture override below keeps the general clean frame unchanged.
    case("Range exclusive " + field, rule("data_quality", "range", params), lambda d, v=value: change(d, "amount", v))
    CASES[-1]["clean"] = lambda: clean_frame().withColumn("amount", F.lit(40.0))
for operator in ("=", "!=", ">", ">=", "<", "<="):
    good = 100.0 if operator == "=" else (120.0 if operator in (">", ">=") else 20.0)
    bad = 100.0 if operator in ("!=", ">", "<") else (120.0 if operator in ("=", "<=" ) else 20.0)
    case("Relationship " + operator, rule("data_quality", "column_relationship", {"columns": ["amount", "ceiling"], "operator": operator}), lambda d, v=bad: change(d, "amount", v))
    CASES[-1]["clean"] = lambda v=good: clean_frame().withColumn("amount", F.lit(v))
for typ in ("conditional_completeness", "conditional_values"):
    params = {"columns": ["category", "required_note" if typ == "conditional_completeness" else "code"], "condition_operator": "!=", "condition_value": "B"}
    params.update({"treat_blank_as_missing": False} if typ == "conditional_completeness" else {"mode": "block", "values": ["ZZ99"]})
    case(typ + " alternate", rule("data_quality", typ, params), lambda d, t=typ: change(d, "required_note" if t == "conditional_completeness" else "code", None if t == "conditional_completeness" else "ZZ99"))
for schema_mode in ("strict", "minimum_required", "relaxed"):
    for mutation, fn in (("missing", lambda d: d.drop("secret")), ("type", lambda d: d.withColumn("amount", F.col("amount").cast("string")))):
        case("Schema " + schema_mode + " " + mutation, rule("schema", schema_mode), fn)
case("Schema strict extra", SCHEMA, lambda d: d.withColumn("unexpected", F.lit(1)))
for mode in ("minimum_required", "relaxed", "skip"):
    case("Schema " + mode + " extra monitor", rule("schema", mode), lambda d: d.withColumn("unexpected", F.lit(1)), expected="monitor")
for unit, age in (("minutes", CLEAN_MAX_AGE_DAYS * 1440), ("hours", CLEAN_MAX_AGE_DAYS * 24), ("days", CLEAN_MAX_AGE_DAYS)):
    case("Freshness " + unit, rule("freshness", "max_age", {"freshness_column": "modified_at", "maximum_age": age, "maximum_age_unit": unit}), lambda d: d.withColumn("modified_at", F.lit(datetime(1900, 1, 1))), boundaries=("Read",))
case("Freshness max_lag_days", rule("freshness", "max_lag_days", {"freshness_column": "modified_at", "max_lag_days": CLEAN_MAX_AGE_DAYS}), lambda d: d.withColumn("modified_at", F.lit(datetime(1900, 1, 1))), boundaries=("Read",))
case("Freshness null latest", FRESH, lambda d: d.withColumn("modified_at", F.lit(None).cast("timestamp")), boundaries=("Read",))
for treatment, definition in SENSITIVE.items():
    # Missing governed column is a legitimate data/schema mutation for every treatment.
    case("Sensitive Data " + treatment, definition, lambda d, c=definition["column"]: d.drop(c), boundaries=("Write",), schema_mode="skip")
case("Sensitive Data Bucket nonnumeric", SENSITIVE["Bucket"], lambda d: d.withColumn("age", F.col("age").cast("string")), boundaries=("Write",), schema_mode="skip")

for item in CASES:
    for boundary in item["boundaries"]:
        role = "source" if boundary == "Read" else "target"
        for action in ("Warn", "Block"):
            definition = item["definition"]
            definitions = [definition] if definition["kind"] == "schema" else [rule("schema", item["schema_mode"]), definition]
            if role == "source" and definition["kind"] != "freshness":
                definitions.append(FRESH)
            key = (item["label"], boundary, action)
            seed_contract(key, role, definitions, action)
print("Frozen contracts:", len(CONTRACTS), "individual scenarios:", len(CASES))


## 03. Baseline tests

The runner snapshots result IDs before every call and accepts only newly written rows for the exact contract version/rule. A matching old activity row cannot make a new test pass. A Block requires an exception **and** a persisted failed rule with `can_continue=False`; arbitrary exceptions are FAIL. Precondition errors are BLOCKED. Unexecuted planned tests remain BLOCKED. DQ failed-value DataFrames and token mappings remain in memory only.

`ARTIFACTS[test_id]` retains full results, frozen payloads, new Metadata rows, captured stage output and traceback. A failure does not stop independent cases. Cleanup is never invoked by this runner.


In [ ]:
from contextlib import contextmanager

@contextmanager
def selection(specs, overrides=None):
    old = FABRIC_CONTEXT.get("data_contract_overrides")
    selected = {OWNED[s["role"]]["table_id"]: {
        "contract_id": s["contract_id"], "contract_version": s["contract_version"]} for s in specs}
    selected.update(overrides or {})
    FABRIC_CONTEXT["data_contract_overrides"] = selected
    try:
        yield
    finally:
        if old is None:
            FABRIC_CONTEXT.pop("data_contract_overrides", None)
        else:
            FABRIC_CONTEXT["data_contract_overrides"] = old

def read_call(role="source"):
    s = OWNED[role]
    return orchestrate_read(store=s["store"], schema=s["schema"], table_name=s["name"],
                            frequency_enabled=False, spark_session=spark, verbose=True)

def write_call(frame, spec, sources=None, mode="validate"):
    s = OWNED[spec["role"]]
    return orchestrate_write(frame, sources=sources or [], store=s["store"], schema=s["schema"],
        table_name=s["name"], write_mode="overwrite", frequency_enabled=False,
        contracts={"tables": {s["table_id"]: {"mode": mode,
            "contract_id": spec["contract_id"], "contract_version": spec["contract_version"]}}},
        spark_session=spark, verbose=True)

def target_version(role="target"):
    return int(DeltaTable.forPath(spark, OWNED[role]["path"]).history(1).select("version").first()[0])

def result_rows(specs):
    table_ids = [OWNED[s["role"]]["table_id"] for s in specs]
    return [r.asDict(recursive=True) for r in metadata("METADATA_GUARDRAIL_RESULTS").where(
        (F.col("environment_name") == ENV) & F.col("table_id").isin(table_ids)).collect()]

def status(value):
    return {"pass": "PASS", "passed": "PASS", "warning": "WARN", "warn": "WARN",
            "failed": "BLOCK", "fail": "BLOCK", "blocked": "BLOCK", "baseline_created": "BASELINE",
            "skipped": "SKIP"}.get(str(value).lower(), str(value).upper())

def expect_rule(spec, kind=None, rule_type=None, expected="PASS"):
    return [(spec, r, expected) for r in spec["rules"]
            if (kind is None or r["guardrail_type"] == kind)
            and (rule_type is None or r["rule_type"] == rule_type)]

def plan(test_id, guardrail, scenario, boundary, specs, expected, evidence):
    row = {"Test ID": test_id, "Guardrail": guardrail, "Test scenario": scenario,
           "Read/Write boundary": boundary,
           "Contract version": json.dumps([{ "id": s["contract_id"], "version": s["contract_version"]} for s in specs]),
           "Expected status": expected, "Actual status": "NOT EXECUTED",
           "Expected rule evidence": json.dumps([{ "rule": r["guardrail_rule_id"], "status": wanted} for s, r, wanted in evidence]),
           "Actual rule evidence": "[]", "PASS/FAIL/SKIP/BLOCKED": "BLOCKED",
           "Diagnostic details": "Planned; execution has not completed."}
    PLAN[test_id] = dict(row)
    RESULTS[test_id] = row

def execute(test_id, guardrail, scenario, boundary, specs, expected, evidence, operation,
            verify=None, unchanged_role=None, block_stage=None, exception_text=None):
    plan(test_id, guardrail, scenario, boundary, specs, expected, evidence)
    row = RESULTS[test_id]
    artifact = {"contracts": [s["payload"] for s in specs], "result": None, "exception": None,
                "traceback": "", "new_results": [], "stage_output": ""}
    ARTIFACTS[test_id] = artifact
    log = io.StringIO()
    try:
        before = {r["guardrail_result_id"] for r in result_rows(specs)}
        previous_version = target_version(unchanged_role) if unchanged_role else None
    except Exception:
        row["Diagnostic details"] = "Precondition failed: " + traceback.format_exc()
        return None
    try:
        try:
            with redirect_stdout(log), redirect_stderr(log):
                artifact["result"] = operation()
        except Exception as exc:
            artifact["exception"] = exc
            artifact["traceback"] = traceback.format_exc()
        artifact["stage_output"] = log.getvalue()
        new_rows = [r for r in result_rows(specs) if r["guardrail_result_id"] not in before]
        artifact["new_results"] = new_rows
        row["Actual rule evidence"] = json.dumps(new_rows, default=str, sort_keys=True)
        error = artifact["exception"]
        result = artifact["result"]
        failed = [r for r in new_rows if r.get("can_continue") is False and status(r.get("status")) == "BLOCK"]
        row["Actual status"] = "BLOCK" if error and failed else ("EXCEPTION" if error else ("WARN" if any(status(r.get("status")) == "WARN" for r in new_rows) else "PASS"))
        problems = []
        if expected == "BLOCK":
            if error is None:
                problems.append("Expected orchestration to raise")
            if not failed:
                problems.append("No newly persisted blocking rule result")
        elif expected == "REJECT":
            chain = []
            current = error
            while current is not None:
                chain.append(current)
                current = current.__cause__
            if not any(isinstance(e, ValueError) for e in chain):
                problems.append("Expected a ValueError in the rejection exception chain")
            if exception_text and exception_text.lower() not in artifact["traceback"].lower():
                problems.append("Rejection reason did not match: " + exception_text)
        elif expected == "COVERAGE_BLOCK":
            if error is None or "Guardrail Coverage" not in str(error):
                problems.append("Expected failure attributed to Guardrail Coverage")
            if "Pipeline is not Guardrail-ready for publication:" not in artifact["traceback"]:
                problems.append("Expected the authoritative coverage readiness rejection")
            if exception_text and exception_text not in artifact["traceback"]:
                problems.append("Wrong coverage rejection reason: " + exception_text)
        elif error is not None:
            problems.append("Unexpected runtime exception")
        if block_stage and (error is None or block_stage not in str(error)):
            problems.append("Wrong blocking stage; expected " + block_stage)
        for spec, rule_doc, wanted in evidence:
            matches = [r for r in new_rows if
                r.get("guardrail_rule_id") == rule_doc["guardrail_rule_id"]
                and r.get("table_id") == OWNED[spec["role"]]["table_id"]
                and r.get("contract_id") == spec["contract_id"]
                and int(r.get("contract_version") or 0) == spec["contract_version"]]
            wanted_set = set(wanted.split("|"))
            if not matches or not any(status(r.get("status")) in wanted_set for r in matches):
                problems.append(f"Missing exact-version {wanted} result for {rule_doc['guardrail_rule_id']}")
            for match in matches:
                canonical = status(match.get("status"))
                if canonical == "BLOCK" and match.get("can_continue") is not False:
                    problems.append("Block result allowed continuation")
                if canonical in {"PASS", "WARN", "BASELINE"} and match.get("can_continue") is not True:
                    problems.append("Passing/warning result disallowed continuation")
        if unchanged_role and target_version(unchanged_role) != previous_version:
            problems.append("Target Delta version changed despite Validate/Block")
        if error is None and result is not None:
            stages = [r["stage"] for r in result.get("orchestration_stages", [])]
            expected_stages = ["Read", "Freshness", "Schema", "Data Quality", "Profile"] if boundary == "Read" else ["Schema", "Sensitive Data", "Source Drift", "Data Quality", "Guardrail Coverage"]
            if boundary == "Write" and result.get("published"):
                expected_stages += ["Write", "Profile"]
            if stages != expected_stages:
                problems.append(f"Stage sequence mismatch: {stages}")
            if verify:
                verify(result, new_rows)
        row["PASS/FAIL/SKIP/BLOCKED"] = "FAIL" if problems else "PASS"
        if expected in {"REJECT", "COVERAGE_BLOCK"} and not problems:
            row["Actual status"] = expected
        row["Diagnostic details"] = "\n".join(problems) + ("\n" + artifact["traceback"] if error else "")
    except Exception:
        row["PASS/FAIL/SKIP/BLOCKED"] = "FAIL"
        row["Diagnostic details"] += "\nAssertion/reporting failure: " + traceback.format_exc()
    print(test_id, row["PASS/FAIL/SKIP/BLOCKED"], row["Actual status"])
    return artifact["result"]

def verify_sensitive(result, treatment, clean):
    prepared = result["dataframe"]
    before = {r["row_id"]: r.asDict() for r in clean.collect()}
    after = {r["row_id"]: r.asDict() for r in prepared.collect()}
    assert before.keys() == after.keys(), "Sensitive treatment changed row identities/count"
    column = SENSITIVE[treatment]["column"]
    if treatment == "Remove":
        assert column not in prepared.columns, "Remove retained the sensitive column"
    elif treatment == "Mask":
        for key in before:
            original = before[key][column]
            expected = None if original is None else (original if len(original) <= 3 else original[0] + "*" * (len(original)-2) + original[-1])
            assert after[key][column] == expected, "Mask content mismatch"
    elif treatment == "Bucket":
        for key in before:
            age = before[key][column]
            expected = None if age is None else ("older" if age >= 60 else "middle" if age >= 30 else "young")
            assert after[key][column] == expected, "Bucket label mismatch"
    else:
        mapping = result["sensitive_result"]["support_mapping"]
        assert mapping is not None, "Token mapping missing"
        lookup = {r["original_value"]: r["token_value"] for r in mapping.collect()}
        originals = {r[column] for r in before.values() if r[column] is not None}
        assert lookup.keys() == originals
        assert len(set(lookup.values())) == len(lookup), "Token collision"
        for key in before:
            original = before[key][column]
            assert after[key][column] == (None if original is None else lookup[original])
            if original is not None:
                assert after[key][column] != original, "Tokenize retained raw value"
    for key in before:
        for other in clean.columns:
            if other != column:
                assert before[key][other] == after[key][other], f"Unrelated column changed: {other}"

plan("baseline/write", "All applicable Guardrails", "Clean Enforce requires successful source Read", "Write",
     [BASE_SOURCE, BASE_TARGET], "PASS", expect_rule(BASE_TARGET) + expect_rule(BASE_SOURCE, "source_drift", expected="BASELINE"))

def assert_validate(result, rows):
    assert result.get("published") is False and result.get("validation_passed") is True
    assert result["coverage_result"]["can_continue"] is True

def assert_published(result):
    assert result.get("published") is True
with selection([BASE_SOURCE, BASE_TARGET]):
    BASE_READ = execute("baseline/read", "All source Guardrails", "Clean Read", "Read", [BASE_SOURCE], "PASS",
                        expect_rule(BASE_SOURCE, "schema") + expect_rule(BASE_SOURCE, "freshness") + expect_rule(BASE_SOURCE, "data_quality"), read_call)
    if BASE_READ is not None:
        BASE_WRITE = execute("baseline/write", "All applicable Guardrails", "Clean Enforce and baseline creation", "Write",
            [BASE_SOURCE, BASE_TARGET], "PASS", expect_rule(BASE_TARGET) + expect_rule(BASE_SOURCE, "source_drift", expected="BASELINE"),
            lambda: write_call(clean_frame(), BASE_TARGET, [BASE_READ], mode="enforce"),
            verify=lambda result, rows: assert_published(result))


## 04. Individual Guardrail tests

For each applicable boundary and both actions: clean baseline → one mutation → restoration → clean replay. The two range fixtures explicitly test exclusive endpoints; relationships cover all six operators, conditional rules cover both condition operators and value modes, and Schema covers strict/relaxed/minimum-required behavior. Freshness covers supported units and legacy lag intent. Sensitive Data uses a missing input column (and nonnumeric Bucket input) to test legitimate treatment failure, plus exact output/mapping checks on clean data.

Monitoring configurations correctly continue with warning even for a Block action. The notebook expects Warn to continue, so an incorrectly blocking Warn implementation appears as FAIL. No assertion is weakened to hide such defects. Guardrail Coverage itself has no Warn setting; its readiness failures are tested separately.


In [ ]:
STAGE_FOR = {"schema": "Schema", "freshness": "Freshness", "data_quality": "Data Quality", "sensitive_data": "Sensitive Data"}
# Predeclare the complete matrix before execution, so an interrupted run cannot look complete.
for index, item in enumerate(CASES):
    for boundary in item["boundaries"]:
        for action in ("Warn", "Block"):
            spec = CONTRACTS[(item["label"], boundary, action)]
            for phase in ("clean", "dirty", "restored"):
                wanted = ("WARN" if action == "Warn" or item["expected"] == "monitor" else "BLOCK") if phase == "dirty" else "PASS"
                test_id = f"case/{index:02}/{boundary}/{action}/{phase}"
                plan(test_id, item["label"], phase, boundary, [spec], wanted,
                     expect_rule(spec, item["definition"]["kind"], expected=wanted))

for index, item in enumerate(CASES):
    for boundary in item["boundaries"]:
        for action in ("Warn", "Block"):
            spec = CONTRACTS[(item["label"], boundary, action)]
            base = item.get("clean", clean_frame)()
            with selection([spec]):
                try:
                    for phase in ("clean", "dirty", "restored"):
                        frame = item["mutate"](base) if phase == "dirty" else base
                        if boundary == "Read":
                            put("source", frame)
                        expected = ("WARN" if action == "Warn" or item["expected"] == "monitor" else "BLOCK") if phase == "dirty" else "PASS"
                        kind = item["definition"]["kind"]
                        checks = expect_rule(spec, kind, expected=expected)
                        # For a clean run all applicable configured rules must actually pass.
                        if phase != "dirty":
                            checks = expect_rule(spec)
                        treatment = item["definition"]["rule_type"].title() if kind == "sensitive_data" else None
                        def verify(result, rows, t=treatment, p=phase, b=base):
                            if boundary == "Write":
                                assert_validate(result, rows)
                                if t and p != "dirty":
                                    verify_sensitive(result, t, b)
                        execute(f"case/{index:02}/{boundary}/{action}/{phase}", item["label"], phase,
                            boundary, [spec], expected, checks,
                            read_call if boundary == "Read" else lambda f=frame: write_call(f, spec),
                            verify=verify, unchanged_role="target" if boundary == "Write" else None,
                            block_stage=STAGE_FOR[kind] if expected == "BLOCK" else None)
                except Exception:
                    # Remaining predeclared phases stay BLOCKED; other scenarios continue.
                    details = traceback.format_exc()
                    ARTIFACTS[f"setup/{index}/{boundary}/{action}"] = {"traceback": details}
                    for phase in ("clean", "dirty", "restored"):
                        pending = RESULTS[f"case/{index:02}/{boundary}/{action}/{phase}"]
                        if pending["PASS/FAIL/SKIP/BLOCKED"] == "BLOCKED":
                            pending["Diagnostic details"] = "Fixture/setup failure: " + details
                finally:
                    if boundary == "Read":
                        put("source", clean_frame())

# Freshness Static is explicitly SKIP, never PASS. It writes no result row in this implementation.
STATIC = seed_contract("static_freshness", "source", [SCHEMA, rule("freshness", "skip", {"refresh_expectation": "static"})])
with selection([STATIC]):
    def verify_static(result, rows):
        assert result["freshness_result"]["status"] == "skipped"
        assert not any(r["guardrail_type"] == "freshness" for r in rows)
    execute("freshness/static", "Freshness", "Static refresh explicitly skips age check", "Read", [STATIC], "PASS",
            expect_rule(STATIC, "schema"), read_call, verify=verify_static)
    RESULTS["freshness/static"]["Diagnostic details"] += "\nFreshness age check was SKIP by configuration; only the skip contract behavior was tested."

# Additional treatment assertions cover nulls, repeat originals and bucket boundaries.
for treatment, definition in SENSITIVE.items():
    spec = seed_contract(("sensitive_edges", treatment), "target", [SCHEMA, definition])
    column = definition["column"]
    edge = change(clean_frame(), column, None, row_id=2)
    if treatment == "Bucket":
        edge = change(change(edge, "age", 30, row_id=1), "age", 60, row_id=3)
    with selection([spec]):
        execute("sensitive/edges/" + treatment, "Sensitive Data " + treatment, "Null preservation, duplicates and boundaries",
                "Write", [spec], "PASS", expect_rule(spec), lambda: write_call(edge, spec),
                verify=lambda result, rows, t=treatment, b=edge: (assert_validate(result, rows), verify_sensitive(result, t, b)),
                unchanged_role="target")


## 05. Combined integration test

Warn must report all deliberately violated DQ rules in one orchestration. Block must stop at the failing stage, with no publication. Downstream checks after that stage are not required to execute. Source Drift uses real source reads and successful writes to establish accepted Source Observation history; it never injects history rows. Each action owns a separate target baseline.


In [ ]:
for action in ("Warn", "Block"):
    for boundary in ("Read", "Write"):
        role = "source" if boundary == "Read" else "target"
        spec = seed_contract(("combined", boundary, action), role,
            [SCHEMA, DQ_DEFS["Completeness"], DQ_DEFS["Range"], DQ_DEFS["Pattern"]] + ([FRESH] if boundary == "Read" else []), action)
        dirty = MUTATIONS["Pattern"](MUTATIONS["Range"](MUTATIONS["Completeness"](clean_frame())))
        expected = "WARN" if action == "Warn" else "BLOCK"
        with selection([spec]):
            try:
                for phase, frame in (("dirty", dirty), ("restored", clean_frame())):
                    wanted = expected if phase == "dirty" else "PASS"
                    checks = expect_rule(spec, "data_quality", expected=wanted)
                    if boundary == "Read":
                        put("source", frame)
                    execute(f"combined/{boundary}/{action}/{phase}", "Combined DQ", phase, boundary,
                        [spec], wanted, checks, read_call if boundary == "Read" else lambda f=frame: write_call(f, spec),
                        unchanged_role="target" if boundary == "Write" else None,
                        block_stage="Data Quality" if wanted == "BLOCK" else None)
            finally:
                put("source", clean_frame())

for action in ("Warn", "Block"):
    role = "drift_" + action.lower()
    reserve(role, "Silver")
    dst = seed_contract(("drift_target", action), role, [SCHEMA])
    src = seed_contract(("drift_source", action), "source", [SCHEMA, FRESH, DRIFT], action,
                        processing={"load_strategy": "append"})
    reset_src = seed_contract(("drift_reset", action), "source", [SCHEMA, FRESH, DRIFT], "Block",
                              processing={"load_strategy": "overwrite"})
    for suffix, planned_spec, expected_status in (
        ("read", src, "PASS"), ("baseline", src, "PASS"), ("dirty-read", src, "PASS"),
        ("mutation", src, "WARN" if action == "Warn" else "BLOCK"),
        ("reset-read", reset_src, "PASS"), ("reset-write", reset_src, "PASS"),
        ("restored-read", src, "PASS"), ("restored", src, "PASS")):
        plan(f"drift/{action}/{suffix}", "Source Drift", suffix,
             "Read" if suffix.endswith("read") else "Write", [planned_spec, dst], expected_status, [])
    with selection([src, dst]):
        put("source", clean_frame())
        read_result = execute(f"drift/{action}/read", "Source Drift", "Read clean source observation", "Read",
                             [src], "PASS", expect_rule(src, "schema") + expect_rule(src, "freshness"), read_call)
        if read_result is not None:
            execute(f"drift/{action}/baseline", "Source Drift", "First accepted source-target baseline", "Write", [src, dst],
                    "PASS", expect_rule(src, "source_drift", expected="BASELINE") + expect_rule(dst),
                    lambda: write_call(clean_frame(), dst, [read_result], mode="enforce"))
        try:
            # Increase historical content while retaining the partition and changing the change timestamp.
            dirty = clean_frame().withColumn("modified_at", F.lit(FIXED_TIME + timedelta(seconds=1)))
            put("source", dirty)
            observed = execute(f"drift/{action}/dirty-read", "Source Drift", "Read mutated source observation", "Read",
                [src], "PASS", expect_rule(src, "schema") + expect_rule(src, "freshness"), read_call)
            if observed is not None:
                wanted = "WARN" if action == "Warn" else "BLOCK"
                execute(f"drift/{action}/mutation", "Source Drift", "Historical mutation incompatible with Append", "Write", [src, dst],
                    wanted, expect_rule(src, "source_drift", expected=wanted),
                    lambda: write_call(clean_frame(), dst, [observed], mode="enforce"),
                    unchanged_role=role if action == "Block" else None,
                    block_stage="Source Drift" if action == "Block" else None)
        finally:
            put("source", clean_frame())
        # A Warn Enforce run accepts the mutation. Restore the accepted baseline through
        # an independently frozen Overwrite contract and a real successful write.
        with selection([reset_src, dst]):
            restored = execute(f"drift/{action}/reset-read", "Source Drift", "Read restored fixture", "Read", [reset_src],
                "PASS", expect_rule(reset_src, "schema") + expect_rule(reset_src, "freshness"), read_call)
            if restored is not None:
                execute(f"drift/{action}/reset-write", "Source Drift", "Overwrite reconciles historical restoration", "Write",
                    [reset_src, dst], "PASS", expect_rule(reset_src, "source_drift"),
                    lambda: write_call(clean_frame(), dst, [restored], mode="enforce"))
        restored = execute(f"drift/{action}/restored-read", "Source Drift", "Read restored Append fixture", "Read", [src],
                           "PASS", expect_rule(src, "schema") + expect_rule(src, "freshness"), read_call)
        if restored is not None:
            execute(f"drift/{action}/restored", "Source Drift", "Clean Append replay after restoration", "Write", [src, dst],
                "PASS", expect_rule(src, "source_drift"), lambda: write_call(clean_frame(), dst, [restored]), unchanged_role=role)


## 06. Contract lifecycle checks

Old frozen versions remain explicitly selectable even after newer versions exist. Both Validate and Enforce run Guardrails; only Enforce may publish. Validate and expected Block/rejection tests compare the target Delta version before/after. Missing, malformed, nonexistent, foreign-table and draft versions must be rejected through the actual orchestrator.

Coverage positive verifies every applicable configured rule was evaluated. Negative cases omit one participant contract or deliberately switch its frozen version after Read, leaving no Read results for the new rule identities. Coverage has a fixed blocking decision, independent of individual rule actions.


In [ ]:
def coverage_pass(result, rows):
    assert_validate(result, rows)
    coverage = result["coverage_result"]
    assert coverage["status"] == "passed" and not coverage["missing"] and not coverage["issues"]
    assert coverage["coverage"] and all(r["evaluated"] for r in coverage["coverage"])

plan("lifecycle/coverage-pass", "Guardrail Coverage", "Requires successful source Read", "Write",
     [BASE_SOURCE, BASE_TARGET], "PASS", expect_rule(BASE_TARGET))
with selection([BASE_SOURCE, BASE_TARGET]):
    source = execute("lifecycle/coverage-read", "Guardrail Coverage", "Refresh source results", "Read", [BASE_SOURCE], "PASS",
                     expect_rule(BASE_SOURCE, "schema") + expect_rule(BASE_SOURCE, "freshness") + expect_rule(BASE_SOURCE, "data_quality"), read_call)
    if source is not None:
        execute("lifecycle/coverage-pass", "Guardrail Coverage", "All configured participants evaluated", "Write", [BASE_SOURCE, BASE_TARGET],
                "PASS", expect_rule(BASE_TARGET) + expect_rule(BASE_SOURCE, "source_drift"),
                lambda: write_call(clean_frame(), BASE_TARGET, [source]), verify=coverage_pass, unchanged_role="target")

# Drift deliberately has no selected contract for this participant; it skips and
# the fixed Coverage gate must attribute the missing contract before publication.
with selection([BASE_TARGET]):
    execute("coverage/missing-contract", "Guardrail Coverage", "Source participant has no selected contract", "Write", [BASE_TARGET],
        "COVERAGE_BLOCK", expect_rule(BASE_TARGET),
        lambda: write_call(clean_frame(), BASE_TARGET, [{"table_id": UNREAD["table_id"]}]), unchanged_role="target", exception_text="no selected Data Contract")

LATE_SOURCE = seed_contract("late_source", "source", [SCHEMA, FRESH, DRIFT, DQ_DEFS["Range"]], "Warn",
                            processing={"load_strategy": "append"})
# The current source observation is real, but these new source Read rules have never run.
with selection([LATE_SOURCE, BASE_TARGET]):
    execute("coverage/missing-results", "Guardrail Coverage", "Select new source version after Read", "Write", [LATE_SOURCE, BASE_TARGET],
        "COVERAGE_BLOCK", expect_rule(BASE_TARGET) + expect_rule(LATE_SOURCE, "source_drift"),
        lambda: write_call(clean_frame(), BASE_TARGET, [{"table_id": SOURCE["table_id"]}]), unchanged_role="target", exception_text="not evaluated")

for action in ("Warn", "Block"):
    spec = CONTRACTS[("Range", "Write", action)]
    for mode in ("validate", "enforce"):
        with selection([spec]):
            wanted = "WARN" if action == "Warn" else "BLOCK"
            execute(f"lifecycle/{action}/{mode}", "Validate versus Enforce", "Same failing Range rule", "Write", [spec],
                wanted, expect_rule(spec, "data_quality", expected=wanted),
                lambda m=mode: write_call(MUTATIONS["Range"](clean_frame()), spec, mode=m),
                verify=(assert_validate if mode == "validate" else lambda result, rows: assert_published(result)),
                unchanged_role="target" if mode == "validate" or action == "Block" else None,
                block_stage="Data Quality" if action == "Block" else None)
    # Old exact version resolves, with rule identities proving the selected action.
    with selection([spec]):
        execute("lifecycle/exact/" + action, "Frozen exact version", "Select older frozen version after later freezes", "Write",
                [spec], "PASS", expect_rule(spec), lambda: write_call(clean_frame(), spec),
                verify=assert_validate, unchanged_role="target")

# Authoritative service creates an open draft; do not freeze it for this rejection test.
DRAFT = create_contract_draft(table_id=TARGET["table_id"], config=CONFIG, env=ENV, spark_session=spark,
    context=FABRIC_CONTEXT, schema=metadata_table_physical_schema(CONFIG, "METADATA_DATA_CONTRACT"))
REJECTIONS = [
    ("missing-version", {"contract_id": BASE_TARGET["contract_id"]}, "requires both"),
    ("missing-id", {"contract_version": BASE_TARGET["contract_version"]}, "requires both"),
    ("invalid-version", {"contract_id": BASE_TARGET["contract_id"], "contract_version": "invalid"}, "integer"),
    ("nonexistent-version", {"contract_id": BASE_TARGET["contract_id"], "contract_version": 999999}, "does not exist"),
    ("nonexistent-contract", {"contract_id": str(uuid.uuid4()), "contract_version": 1}, "does not exist"),
    ("draft-version", {"contract_id": DRAFT["contract_id"], "contract_version": DRAFT["contract_version"]}, "must be frozen"),
    ("foreign-table", {"contract_id": BASE_SOURCE["contract_id"], "contract_version": BASE_SOURCE["contract_version"]}, "does not belong"),
]
for label, override, message in REJECTIONS:
    with selection([], {TARGET["table_id"]: override}):
        execute("lifecycle/reject/" + label, "Contract rejection", label, "Write", [BASE_TARGET], "REJECT", [],
                lambda: write_call(clean_frame(), BASE_TARGET), unchanged_role="target", exception_text=message)

# Immutability check reads every seeded version after all execution; no patching or repair.
FROZEN_AFTER = {(r["contract_id"], int(r["contract_version"])): r.asDict() for r in
    metadata("METADATA_DATA_CONTRACT").where(F.col("table_id").isin([s["table_id"] for s in OWNED.values()])).collect()}
for key, spec in CONTRACTS.items():
    record = spec["record"]
    actual = FROZEN_AFTER.get((spec["contract_id"], spec["contract_version"]), {})
    test_id = "lifecycle/immutable/" + str(key)
    plan(test_id, "Frozen immutability", "Saved definition unchanged after execution", "Metadata", [spec], "PASS", [])
    intact = actual.get("status") == "frozen" and actual.get("contract_payload_json") == record["contract_payload_json"]
    RESULTS[test_id].update({"Actual status": "PASS" if intact else "FAIL", "PASS/FAIL/SKIP/BLOCKED": "PASS" if intact else "FAIL",
        "Diagnostic details": "Compared persisted frozen payload to freeze-service return value."})


## 07. Manual UI checklist (optional reference)

<details><summary>Equivalent widget_data_contract() authoring checklist</summary>

For every item: select the test-owned table in Development, create/open a draft, select the exact column identities/types from Catalogue, author the rule, choose Warn or Block, Save and Freeze. Choose that exact frozen version in `widget_select_data_contract()` before Read/Write. Never edit a frozen row. The notebook does all of this programmatically; the UI is not a prerequisite.

- [ ] **Freshness:** choose `modified_at`, maximum age and minutes/hours/days (or lag days); choose static/no refresh for the skip case. Replace all timestamps with stale values, then restore.
- [ ] **Schema:** select Strict, Minimum Required, Relaxed or monitoring/Skip; review all frozen column types. Remove a column, change one type, or add an unexpected column as separate tests.
- [ ] **Source Drift:** set `partition_day` and `modified_at`, select Append processing; Read then Enforce Write to establish the source-target baseline. Change an already accepted partition and Read/Write again. Overwrite processing can reconcile the restoration.
- [ ] **Completeness:** select `name`, zero maximum missing percent; toggle blank-as-missing. Test null and blank separately.
- [ ] **Uniqueness:** select `row_id`, 100% minimum uniqueness; additionally select composite `category,code`. Duplicate the configured key.
- [ ] **Whitelist:** add a DQ allowed-values rule on `category` with A and B; introduce X.
- [ ] **Blacklist:** add a separate DQ blocked-values rule on `category` with DENY; introduce DENY.
- [ ] **Range:** select `amount`, 0–100 inclusive; test -1. Also author exclusive lower/upper bounds and test values exactly on those bounds.
- [ ] **Pattern:** author the format intent for `code` (two uppercase letters and two digits); review the resolved Pattern `^[A-Z]{2}[0-9]{2}$`; introduce `invalid`.
- [ ] **Column Relationship:** compare `amount` with `ceiling`; choose each supported comparison operator; adjust clean and dirty values as shown in CASES.
- [ ] **Conditional Completeness:** when `category=A`, require `required_note`; configure blank handling. Test `!=B` and null handling separately.
- [ ] **Conditional Values:** when `category=A`, allow `code` AA01/CC03; separately author `!=B` with a blocked ZZ99 value. Introduce ZZ99.
- [ ] **Custom Expression:** author a PySpark boolean expression `F.col('amount') >= F.lit(0)`; introduce -1.
- [ ] **Sensitive Data Mask:** select `email`, preserve one leading/trailing character, mask character `*`; inspect returned prepared DataFrame.
- [ ] **Sensitive Data Tokenize:** select `email`, Tokenize; inspect caller-owned mapping without persisting it. Repeated originals must share a token.
- [ ] **Sensitive Data Bucket:** select numeric `age`, boundaries 0/30/60 and labels young/middle/older; test exact endpoints and nulls.
- [ ] **Sensitive Data Remove:** select `secret`, Remove; verify the prepared DataFrame has no such column.
- [ ] **Sensitive failure actions:** use Schema monitoring to allow a missing governed treatment column to reach Sensitive Data; use Warn then Block frozen versions. Bucket additionally rejects nonnumeric data.
- [ ] **Guardrail Coverage:** select contracts for all participants and execute all applicable source/target checks. Omit a participant contract or switch its contract after Read to reproduce coverage failures. Coverage is a fixed Block gate, with no Warn selector.
- [ ] **Validate/Enforce:** choose Validate to run checks without publication; choose Enforce to publish after checks. Confirm target Delta history and exact contract version.

</details>


## 08. Results summary

The consolidated DataFrame reports PASS/FAIL/SKIP/BLOCKED independently from the expected runtime status. A deliberate, evidenced Block is PASS. Planned cases without a completed execution remain BLOCKED. A configured static Freshness skip is explicitly identified in diagnostics; it never proves an age check passed.

Inspect `ARTIFACTS[test_id]['result']`, `['new_results']`, `['stage_output']`, `['exception']` and `['traceback']` to investigate a scenario. A failed orchestrator does not return partial stage records, so its captured stage log and persisted results are retained instead. To rerun an individual case, select its contract and call `execute()` with its original fixture/mutation; test IDs replace their prior summary row while the frozen version remains immutable. Do not treat a defect as a reason to change expected statuses. Fix authoritative production functions separately.

Runtime cost: this exhaustive matrix makes many Spark/Delta operations over three synthetic records. Run on a maintainer Development workspace, not a shared production job.


In [ ]:
SUMMARY_COLUMNS = ["Test ID", "Guardrail", "Test scenario", "Read/Write boundary", "Contract version", "Expected status", "Actual status", "Expected rule evidence", "Actual rule evidence", "PASS/FAIL/SKIP/BLOCKED", "Diagnostic details"]
summary_df = spark.createDataFrame([tuple(str(row[column]) for column in SUMMARY_COLUMNS)
                                     for row in RESULTS.values()], SUMMARY_COLUMNS)
display(summary_df.orderBy("Test ID"))
display(summary_df.groupBy("PASS/FAIL/SKIP/BLOCKED").count())
print("FabricOps version:", __version__, "Namespace:", RUN_NAMESPACE)
print("Observed summary counts:", {r[0]: r[1] for r in summary_df.groupBy("PASS/FAIL/SKIP/BLOCKED").count().collect()})
print("Review failures before cleanup. Artifacts retained in memory:", len(ARTIFACTS))
print("For cleanup set CLEANUP_REVIEWED_NAMESPACE to:", RUN_NAMESPACE)
# Restore the caller's selection after tests; keep the notebook's owned manifest.
FABRIC_CONTEXT.clear()
FABRIC_CONTEXT.update(BASE_CONTEXT)
builtins.FABRIC_CONTEXT = FABRIC_CONTEXT


## 09. Cleanup

**Explicit opt-in after review only.** Set the switch below to the current printed run namespace, then run this cell. The default is inert during Run all. Exact owned table IDs and resolved paths are the deletion manifest; prefix scans are never used to find deletion targets. Metadata predicates always include Development and exact seeded identities. Source Observation rows require both owned source and owned target. No global activity purge, DROP SCHEMA, wildcard deletion, contract activation or VACUUM occurs.

Cleanup removes all rows for the exclusively owned table identities, including any partially created drafts, profiles, lineage, Guardrails and results. It then removes only the known table directories. Retrying uses the same manifest and is safe after partial completion. It does not provision or drop shared metadata tables. Current authoritative metadata schemas do not contain `METADATA_GUARDRAIL_ROW_RESULTS`; failed-value frames are caller-owned and no nonexistent table is invented.

Results and diagnostics stay in memory for review; token mapping caches are unpersisted. Do not restart the session before cleanup: ownership is deliberately notebook-local, not stored in another fixture file.


In [ ]:
CLEANUP_REVIEWED_NAMESPACE = ""  # After review, paste RUN_NAMESPACE here and run this cell explicitly.
if CLEANUP_REVIEWED_NAMESPACE != RUN_NAMESPACE:
    print("Cleanup not requested. Review section 08; then set the exact namespace in this cell.")
else:
    assert ENV == "dev" and OWNED and re.fullmatch(r"fo_gr_ut_[0-9a-f]{16}", RUN_NAMESPACE)
    ids = [s["table_id"] for s in OWNED.values()]
    assert len(ids) == len(set(ids)), "Ownership manifest has duplicate identities"
    for role, spec in OWNED.items():
        assert spec["name"] == RUN_NAMESPACE + "_" + role
        _, _, _, resolved_path = resolve_configured_lakehouse_table(
            spec["store"], spec["name"], spec["schema"], context=FABRIC_CONTEXT)
        assert resolved_path == spec["path"], "Configuration changed; refusing cleanup"
    # Validate every deletion predicate before the first mutation.
    deletions = []
    for name in META_NAMES:
        frame = metadata(name)
        columns = set(frame.columns)
        if name == "METADATA_SOURCE_OBSERVATION":
            owned_predicate = F.col("source_table_id").isin(ids) & F.col("target_table_id").isin(ids)
        else:
            assert "table_id" in columns, f"No exact ownership column for {name}; inspect before cleanup"
            owned_predicate = F.col("table_id").isin(ids)
        predicate = (F.col("environment_name") == "dev") & owned_predicate
        _, _, _, path = resolve_configured_lakehouse_table("Metadata", name,
            metadata_table_physical_schema(CONFIG, name), context=FABRIC_CONTEXT)
        deletions.append((name, path, predicate))
    for name, path, predicate in deletions:
        DeltaTable.forPath(spark, path).delete(predicate)
        assert metadata(name).where(predicate).limit(1).count() == 0, f"Cleanup incomplete: {name}"
        print("Removed exact owned rows from", name)
    for spec in OWNED.values():
        hpath = spark._jvm.org.apache.hadoop.fs.Path(spec["path"])
        fs = hpath.getFileSystem(spark._jsc.hadoopConfiguration())
        if fs.exists(hpath):
            assert fs.delete(hpath, True), f"Could not remove owned table {spec['name']}"
        assert not fs.exists(hpath)
        print("Removed owned table", spec["name"])
    for artifact in ARTIFACTS.values():
        result = artifact.get("result") or {}
        mapping = result.get("sensitive_result", {}).get("support_mapping")
        if mapping is not None:
            mapping.unpersist()
    FABRIC_CONTEXT.clear()
    FABRIC_CONTEXT.update(BASE_CONTEXT)
    builtins.FABRIC_CONTEXT = ORIGINAL_CONTEXT if ORIGINAL_CONTEXT is not None else FABRIC_CONTEXT
    CLEANED = True
    print("Cleanup complete for", RUN_NAMESPACE, "(safe to rerun this cell).")
